# Lab 1 — Triple Extraction

**Goal:** Turn unstructured text into structured (subject, predicate, object) triples using an LLM with structured outputs.

**Why triples?**  
Triples are the atomic unit of knowledge. Every fact can be expressed as: _someone_ → _relationship_ → _something_.  
Build enough triples and you have a knowledge graph.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from dotenv import load_dotenv
load_dotenv(override=True)
from knowledge_graph import extract_triples
print('Ready ✓')

## 1. Extract from a simple paragraph

In [ ]:
TEXT = """
Jeff Bezos founded Amazon in 1994 in Seattle. Amazon later acquired Whole Foods in 2017.
Andy Jassy became CEO of Amazon in 2021. Amazon Web Services, a subsidiary of Amazon, 
is headquartered in Seattle. AWS competes with Microsoft Azure and Google Cloud.
"""

triples = extract_triples(TEXT)

print(f'Extracted {len(triples)} triples:\n')
for t in triples:
    print(f'  ({t.subject}) --[{t.predicate}]--> ({t.obj})  [conf={t.confidence:.0%}]')

## 2. Explore predicate vocabulary

In [ ]:
from collections import Counter
predicates = Counter(t.predicate for t in triples)
print('Predicates found:')
for pred, count in predicates.most_common():
    print(f'  {pred}: {count}')

## 3. Confidence filtering
Low-confidence triples should be reviewed before adding to the graph.

In [ ]:
high_conf = [t for t in triples if t.confidence >= 0.9]
low_conf  = [t for t in triples if t.confidence < 0.9]

print(f'High confidence (≥90%): {len(high_conf)}')
print(f'Low confidence  (<90%): {len(low_conf)}')
if low_conf:
    print('\nLow confidence triples (review before adding):')
    for t in low_conf:
        print(f'  [{t.confidence:.0%}] ({t.subject}) --[{t.predicate}]--> ({t.obj})')

## 4. Extract from multiple documents and merge

In [ ]:
DOC2 = """
Sundar Pichai is the CEO of Alphabet, the parent company of Google.
Google was founded by Larry Page and Sergey Brin at Stanford University in 1998.
Google Cloud competes with AWS and Azure in the cloud market.
"""

triples2 = extract_triples(DOC2)
all_triples = triples + triples2

print(f'Doc 1: {len(triples)} triples')
print(f'Doc 2: {len(triples2)} triples')
print(f'Total: {len(all_triples)} triples')

# Find shared entities (entities mentioned in both docs)
entities1 = {t.subject for t in triples} | {t.obj for t in triples}
entities2 = {t.subject for t in triples2} | {t.obj for t in triples2}
shared = entities1 & entities2
print(f'\nShared entities (bridge nodes): {shared}')

## Exercise
Extend the extraction to also extract **temporal facts** — triples that include a year or date.  
Add a `year: Optional[int]` field to the Triple model and populate it when the text mentions a specific year.  
Test on text with multiple dated events.

In [ ]:
# Your code here
